In [1]:
import numpy as np
import pandas as pd



In [2]:
# Prefer the competition dataset path; fallback to the flat /kaggle/input files if needed.
TRAIN_PATH = "/kaggle/input/tabular-playground-series-dec-2021/train.csv"
TEST_PATH = "/kaggle/input/tabular-playground-series-dec-2021/test.csv"
SAMPLE_PATH = "/kaggle/input/tabular-playground-series-dec-2021/sample_submission.csv"

train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)
sample_sub = pd.read_csv(SAMPLE_PATH)



In [3]:
X = train_df.drop("Cover_Type", axis=1)
y = train_df["Cover_Type"]

# Bugfix: XGBClassifier expects classes [0..K-1] by default; dataset labels are [1..7].
# Keep core logic identical, just shift labels for training then shift back for submission.
y_xgb = y.astype(np.int32) - 1

X.shape, y.shape, y_xgb.min(), y_xgb.max()



((3600000, 55), (3600000,), 0, 6)

In [4]:
from sklearn.model_selection import train_test_split

# Keep the same split settings as the original code (very tiny val set).
x_train, x_val, y_train, y_val = train_test_split(
    X, y_xgb, test_size=0.00001, random_state=42
)
x_train.shape, x_val.shape, y_train.shape, y_val.shape



((3599964, 55), (36, 55), (3599964,), (36,))

In [5]:
from xgboost import XGBClassifier

# Bugfix: use CPU-compatible tree_method so it runs on Kaggle without GPU.
# Core model choice remains XGBClassifier with the same key hyperparameters.
model_xgbc = XGBClassifier(
    n_estimators=118,
    n_jobs=4,
    learning_rate=0.5,
    tree_method="hist",
    random_state=42,
)

model_xgbc.fit(x_train, y_train, eval_set=[(x_val, y_val)], verbose=True)



[0]	validation_0-mlogloss:0.74580
[1]	validation_0-mlogloss:0.50114
[2]	validation_0-mlogloss:0.33981
[3]	validation_0-mlogloss:0.26886
[4]	validation_0-mlogloss:0.20255
[5]	validation_0-mlogloss:0.17375
[6]	validation_0-mlogloss:0.14895
[7]	validation_0-mlogloss:0.13383
[8]	validation_0-mlogloss:0.12161
[9]	validation_0-mlogloss:0.11359
[10]	validation_0-mlogloss:0.10325
[11]	validation_0-mlogloss:0.09218
[12]	validation_0-mlogloss:0.08886
[13]	validation_0-mlogloss:0.08492
[14]	validation_0-mlogloss:0.08035
[15]	validation_0-mlogloss:0.07647
[16]	validation_0-mlogloss:0.07682
[17]	validation_0-mlogloss:0.07419
[18]	validation_0-mlogloss:0.07126
[19]	validation_0-mlogloss:0.06834
[20]	validation_0-mlogloss:0.06610
[21]	validation_0-mlogloss:0.06399
[22]	validation_0-mlogloss:0.06403
[23]	validation_0-mlogloss:0.06284
[24]	validation_0-mlogloss:0.05994
[25]	validation_0-mlogloss:0.05959
[26]	validation_0-mlogloss:0.05989
[27]	validation_0-mlogloss:0.05979
[28]	validation_0-mlogloss:0.0

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              gamma=None, grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.5, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=None, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=118, n_jobs=4,
              num_parallel_tree=None, objective='multi:softprob', ...)

In [6]:
# Predict 0..6, then convert back to 1..7 for submission.
y_predict_xgbc_0based = model_xgbc.predict(test_df)
y_predict_xgbc = y_predict_xgbc_0based.astype(np.int32) + 1

y_predict_xgbc[:10], y_predict_xgbc.min(), y_predict_xgbc.max()



(array([1, 2, 1, 1, 7, 1, 2, 1, 1, 2], dtype=int32), 1, 7)

In [7]:
# Bugfix: build submission from sample_submission to guarantee correct columns and order.
result = sample_sub.copy()
result["Cover_Type"] = y_predict_xgbc
result.head()



,Id,Cover_Type
0,814683,1
1,1357371,2
2,2106112,1
3,3483684,1
4,3960754,7


In [8]:
result.shape, result.columns.tolist()



((400000, 2), ['Id', 'Cover_Type'])

In [9]:
# Ensure a valid .csv submission file is produced.
OUT_PATH = "/kaggle/working/submission.csv"
result.to_csv(OUT_PATH, index=False)
print("Done. Wrote:", OUT_PATH)
print(result.head())

Done. Wrote: /kaggle/working/submission.csv
        Id  Cover_Type
0   814683           1
1  1357371           2
2  2106112           1
3  3483684           1
4  3960754           7
